In [8]:
import pandas
import torch
from hugsvision.dataio.VisionDataset import VisionDataset
from hugsvision.nnet.VisionClassifierTrainer import VisionClassifierTrainer
from transformers import ViTFeatureExtractor, ViTForImageClassification, AutoImageProcessor
from hugsvision.inference.VisionClassifierInference import VisionClassifierInference

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

In [9]:
# Quick sanity print to confirm CUDA availability and the detected GPU model
print("CUDA disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA disponible: True
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [10]:
#data preaparatoion 
train, val, id2label, label2id = VisionDataset.fromImageFolder(
    "./train/",
    test_ratio=0.1,
    balanced=True,
    augmentation=True,
    torch_vision=False
)

Split Datasets...
Balance train dataset...
The less represented label in train as 100 occurrences
Size of train after balancing is 300
Training Dataset Elements:  270
+---------+---------------+-------+-------+-------+
| Dataset | affenpinscher | akita | corgi | Total |
+---------+---------------+-------+-------+-------+
|  Train  |      89       |  91   |  90   |  270  |
|  Test   |      11       |   9   |  10   |  30   |
+---------+---------------+-------+-------+-------+


In [11]:
huggingface_model = 'google/vit-base-patch16-224-in21k'

In [12]:
model = ViTForImageClassification.from_pretrained(
    huggingface_model,
    num_labels=len(label2id),
    label2id=label2id,
    id2label=id2label
)

processor = AutoImageProcessor.from_pretrained(huggingface_model)  # evita warning

trainer = VisionClassifierTrainer(
    model_name   = "a9zin",
    train        = train,
    test         = val,
    output_dir   = "./out/",
    max_epochs   = 20,
    batch_size   = 4,
    lr           = 2e-5,
    fp16         = True,
    model        = model,
    feature_extractor = processor,    
    eval_metric  = "eval_loss",     
)


Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Fast image processor class <class 'transformers.models.vit.image_processing_vit_fast.ViTImageProcessorFast'> is available for this model. Using slow image processor class. To use the fast image processor class set `use_fast=True`.


{'0': 'affenpinscher', '1': 'akita', '2': 'corgi'}
{'affenpinscher': '0', 'akita': '1', 'corgi': '2'}
Trainer builded!
Start Training!


Epoch,Training Loss,Validation Loss
1,No log,0.350167
2,No log,0.110412
3,No log,0.068396
4,No log,0.053194
5,No log,0.046225
6,No log,0.042183
7,No log,0.039810
8,0.145200,0.037672
9,0.145200,0.037325
10,0.145200,0.037085


Model saved at: ./out/A9ZIN/20_2025-09-28-01-03-43


d
